# DOCX Preprocessing Test

브런치 `preprocessing/v2/sumin`에서 추가한 **DOCX → Markdown 추출 파이프라인** 테스트 노트북.

대상 모듈: `structverify.preprocessing.docx`
- `models.py` — `DocxParagraph`, `DocxTable`, `DocxExtracted`
- `reader.py` — `parse_docx`, `docling_extract`
- `markdown.py` — `to_markdown`
- `pipeline.py` — `extract_docx_to_markdown` (공개 API)
- `extractor.py` — `extract_text` 디스패처 연결

## 테스트 시나리오
1. 환경 / import 확인
2. 샘플 DOCX 생성 (python-docx로 인메모리 작성)
3. `reader.parse_docx` — 단락/테이블/메타데이터 파싱
4. `reader.docling_extract` — Docling 보강 (선택, 미설치 시 skip)
5. `markdown.to_markdown` — 직렬화 결과 확인
6. `extract_docx_to_markdown` — 전체 파이프라인
7. `extract_text` 디스패처 연동
8. 사용자 본인의 DOCX 파일로 검증

## 1. 환경 / import 확인

노트북이 `backend/colab/`에 있으므로 프로젝트 루트(`backend/`)를 `sys.path`에 추가해 `structverify` 패키지를 import 가능하게 만든다.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "colab" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("exists structverify/:", (PROJECT_ROOT / "structverify").is_dir())

In [ ]:
# 필요한 패키지 확인
import importlib

for mod in ["docx", "docling"]:
    try:
        m = importlib.import_module(mod)
        print(f"OK   {mod}: {getattr(m, '__version__', '?')}")
    except ImportError as e:
        print(f"MISS {mod}: {e}")
        print(f"     → 설치: pip install {'python-docx' if mod == 'docx' else 'docling'}")

In [ ]:
# 테스트 대상 모듈 import
from structverify.preprocessing.docx import extract_docx_to_markdown
from structverify.preprocessing.docx import reader, markdown as md_mod
from structverify.preprocessing.docx.models import DocxExtracted, DocxParagraph, DocxTable
from structverify.preprocessing.extractor import extract_text
from structverify.core.schemas import SourceType

print("imports OK")

## 2. 샘플 DOCX 생성

외부 파일 없이 노트북만으로 돌릴 수 있도록 `python-docx`로 샘플 문서를 만든다.

포함 요소:
- core_properties.title / created
- Heading 1, Heading 2 (헤딩 레벨 감지 검증)
- 일반 단락 (Normal)
- 리스트 (List Bullet, List Number)
- 테이블 (헤더 + 데이터 행)

In [ ]:
import datetime
import tempfile
import docx

def make_sample_docx() -> str:
    doc = docx.Document()

    # 메타데이터
    doc.core_properties.title = "샘플 보고서"
    doc.core_properties.created = datetime.datetime(2026, 5, 7)

    # 헤딩
    doc.add_heading("개요", level=1)
    doc.add_paragraph("이 문서는 DOCX 추출 파이프라인 테스트용입니다.")

    doc.add_heading("세부 항목", level=2)
    doc.add_paragraph("항목 1", style="List Bullet")
    doc.add_paragraph("항목 2", style="List Bullet")
    doc.add_paragraph("첫 번째 단계", style="List Number")
    doc.add_paragraph("두 번째 단계", style="List Number")

    doc.add_heading("통계 표", level=2)
    table = doc.add_table(rows=3, cols=3)
    headers = ["연도", "매출", "증가율"]
    rows = [
        ["2024", "100", "-"],
        ["2025", "120", "20%"],
    ]
    for j, h in enumerate(headers):
        table.rows[0].cells[j].text = h
    for i, r in enumerate(rows, start=1):
        for j, v in enumerate(r):
            table.rows[i].cells[j].text = v

    doc.add_paragraph("마무리 단락입니다.")

    tmp = tempfile.NamedTemporaryFile(suffix=".docx", delete=False)
    doc.save(tmp.name)
    return tmp.name

SAMPLE_PATH = make_sample_docx()
print("sample saved:", SAMPLE_PATH)

## 3. `reader.parse_docx` — 1차 파싱

검증 포인트:
- title / date 메타데이터
- 단락 헤딩 레벨 (`heading_level`)이 "Heading 1" → 1, "Heading 2" → 2 로 감지되는지
- 리스트 스타일명 보존 (`List Bullet`, `List Number`)
- 테이블 headers + rows 분리
- `source_used == "python-docx"`

In [ ]:
ex: DocxExtracted = reader.parse_docx(SAMPLE_PATH)

print("title       :", repr(ex.title))
print("date        :", repr(ex.date))
print("source_used :", ex.source_used)
print("# paragraphs:", len(ex.paragraphs))
print("# tables    :", len(ex.tables))
print()
for p in ex.paragraphs:
    print(f"  [{p.index}] hl={p.heading_level} style={p.style!r:22} text={p.text!r}")
print()
for t in ex.tables:
    print(f"  table[{t.index}] headers={t.headers} rows={t.rows}")

In [ ]:
# 간단 어서션 — 의도한 결과인지 자동 체크
assert ex.title == "샘플 보고서", f"title mismatch: {ex.title!r}"
assert ex.date == "2026-05-07", f"date mismatch: {ex.date!r}"
assert ex.source_used == "python-docx"

headings = [p for p in ex.paragraphs if p.heading_level is not None]
assert any(p.heading_level == 1 for p in headings), "Heading 1 미감지"
assert any(p.heading_level == 2 for p in headings), "Heading 2 미감지"

assert len(ex.tables) == 1
assert ex.tables[0].headers == ["연도", "매출", "증가율"]
assert ex.tables[0].rows == [["2024", "100", "-"], ["2025", "120", "20%"]]

print("parse_docx: assertions passed")

## 4. `reader.docling_extract` — Docling 보강 (선택)

Docling 미설치 시 `{ok: False, json: None, html: None}` 으로 graceful degrade 되는지 확인.

In [ ]:
dl = reader.docling_extract(SAMPLE_PATH)
print("ok      :", dl["ok"])
print("has json:", dl["json"] is not None)
print("has html:", dl["html"] is not None)

if dl["ok"] and dl["json"]:
    print("docling title:", dl["json"].get("title"))
    print("keys (top 10):", list(dl["json"].keys())[:10])
else:
    print("(Docling 미설치 또는 변환 실패 — 파이프라인은 python-docx 결과만으로 진행)")

## 5. `markdown.to_markdown` — 직렬화

검증 포인트:
- `# 제목` / `_작성일: ..._` 헤더
- 헤딩 레벨이 `#` 개수로 매핑
- `List Bullet` → `- `, `List Number` → `1. ` 으로 직렬화
- 테이블이 문서 말미에 GFM 표로 추가

In [ ]:
md = md_mod.to_markdown(ex)
print(md)

In [ ]:
assert md.startswith("# 샘플 보고서"), "제목 라인 누락"
assert "_작성일: 2026-05-07_" in md, "작성일 라인 누락"
assert "# 개요" in md and "## 세부 항목" in md, "헤딩 매핑 실패"
assert "- 항목 1" in md and "- 항목 2" in md, "불릿 리스트 실패"
assert "1. 첫 번째 단계" in md and "2. 두 번째 단계" in md, "번호 리스트 실패"
assert "| 연도 | 매출 | 증가율 |" in md, "테이블 헤더 누락"
assert "| 2025 | 120 | 20% |" in md, "테이블 데이터 누락"
print("to_markdown: assertions passed")

## 6. `extract_docx_to_markdown` — 전체 파이프라인

공개 API. 단계 3~5를 합친 결과와 동일해야 한다.

In [ ]:
md_pipe = extract_docx_to_markdown(SAMPLE_PATH)
print(md_pipe)
print("---")
print("matches step-by-step result:", md_pipe == md)

In [ ]:
# Edge case: 존재하지 않는 파일 → 빈 문자열 반환 + 에러 로그
result = extract_docx_to_markdown("/nonexistent/path/foo.docx")
print("missing file → result:", repr(result))
assert result == "", "존재하지 않는 파일은 빈 문자열을 반환해야 함"
print("pipeline edge case OK")

## 7. `extract_text` 디스패처

`SourceType.DOCX` 로 들어왔을 때 `_extract_from_docx` → `extract_docx_to_markdown` 으로 라우팅되는지.

In [ ]:
# Jupyter는 이미 이벤트 루프를 돌리고 있으므로 asyncio.run() 대신 top-level await 사용
md_dispatch = await extract_text(SAMPLE_PATH, SourceType.DOCX)
print(md_dispatch[:300] + ("..." if len(md_dispatch) > 300 else ""))
assert md_dispatch == md_pipe, "dispatcher 결과가 pipeline 결과와 다름"
print("dispatcher routing OK")

## 8. 사용자 DOCX 파일로 검증 (선택)

본인의 DOCX 파일이 있으면 아래 `MY_DOCX` 경로를 바꿔서 실행.

In [ ]:
MY_DOCX = ""  # 예: "/Users/me/Documents/sample.docx"

if MY_DOCX and Path(MY_DOCX).is_file():
    ex2 = reader.parse_docx(MY_DOCX)
    print("title       :", ex2.title)
    print("date        :", ex2.date)
    print("# paragraphs:", len(ex2.paragraphs))
    print("# tables    :", len(ex2.tables))
    print()
    md2 = extract_docx_to_markdown(MY_DOCX)
    print("--- Markdown ---")
    print(md2)
else:
    print("MY_DOCX 비어있음 또는 파일 없음 — skip")

## 정리

In [ ]:
import os
try:
    os.unlink(SAMPLE_PATH)
    print("removed:", SAMPLE_PATH)
except FileNotFoundError:
    pass